# Demo & Test

Sources :

- Prices : https://ember-energy.org/data/european-wholesale-electricity-price-data
- Reserves : https://www.services-rte.com/fr/telechargez-les-donnees-publiees-par-rte.html?category=market&type=balancing_capacity&subType=procured_reserves

## 1) Chargement et traitement de la data

J'ai gardé les transformations pour se rappeler de comment on a transformé la data depuis la source brute mais c'est un chantier.

-> A ne pas exécuter à nouveau, tout est dans le dossier data.

In [56]:
PATH = r"C:\Users\thibc\Downloads"

import polars as pl
import pandas as pd
df_prices = pl.read_csv(f"{PATH}/France.csv")
df_reserves = pl.read_excel(f"{PATH}/Reserves.xlsx")

In [35]:
df_prices = df_prices.to_pandas()
df_reserves = df_reserves.to_pandas()

In [ ]:
df_prices = df_prices[["Datetime (UTC)", "Price (EUR/MWhe)"]].rename(
    columns={"Datetime (UTC)": "Datetime", "Price (EUR/MWhe)": "Price"}
)
df_prices.index = pd.to_datetime(df_prices["Datetime"], utc=True)
df_prices.drop(columns=["Datetime"], inplace=True)
df_prices.to_parquet("data/prices.parquet")

In [60]:
df = pd.read_parquet("../data/prices.parquet")
display(df)

,Price
Datetime,
2015-01-01 00:00:00+00:00,36.56
2015-01-01 00:15:00+00:00,36.56
2015-01-01 00:30:00+00:00,36.56
2015-01-01 00:45:00+00:00,36.56
2015-01-01 01:00:00+00:00,36.56
...,...
2025-12-21 05:00:00+00:00,48.96
2025-12-21 05:15:00+00:00,48.96
2025-12-21 05:30:00+00:00,48.96


In [59]:
assert not df.index.duplicated().any()
pd.DataFrame(df).to_parquet("../data/prices.parquet")

In [54]:
df = df["Price"].resample("15min").ffill()

In [55]:
display(df)

Datetime
2015-01-01 00:00:00+00:00    36.56
2015-01-01 00:15:00+00:00    36.56
2015-01-01 00:30:00+00:00    36.56
2015-01-01 00:45:00+00:00    36.56
2015-01-01 01:00:00+00:00    36.56
                             ...  
2025-12-21 05:00:00+00:00    48.96
2025-12-21 05:15:00+00:00    48.96
2025-12-21 05:30:00+00:00    48.96
2025-12-21 05:45:00+00:00    48.96
2025-12-21 06:00:00+00:00    55.72
Freq: 15min, Name: Price, Length: 384697, dtype: float64

In [36]:
display(df_reserves)

,Date,Heures,Type de réserve,Type de produit,Sens de la réserve,Quantité contractualisée (en MW),Prix de la réserve (en euros/MW/15min),Temporalité
0,2025-01-01,00:00 - 00:15,Réserve primaire,STD,A la hausse et à la baisse,670,0.69,Journalier
1,2025-01-01,00:00 - 00:15,Réserve secondaire,STD,A la hausse,713,5.39,Journalier
2,2025-01-01,00:00 - 00:15,Réserve secondaire,STD,A la baisse,755,5.39,Journalier
3,2025-01-01,00:00 - 00:15,Réserve rapide,SPE0,A la hausse,405,0.03,Journalier
4,2025-01-01,00:00 - 00:15,Réserve rapide,SPE0,A la hausse,651,0.31,Annuel
...,...,...,...,...,...,...,...,...
199081,2025-12-14,00:00 - 00:15,Réserve secondaire,STD,A la hausse,701,4.92,Journalier
199082,2025-12-14,00:00 - 00:15,Réserve secondaire,STD,A la baisse,758,3.64,Journalier
199083,2025-12-14,00:00 - 00:15,Réserve rapide,SPE0,A la hausse,393,0.21,Journalier
199084,2025-12-14,00:00 - 00:15,Réserve rapide,SPE0,A la hausse,607,0.31,Périodique


In [37]:
df = df_reserves.copy()

df["Date"] = pd.to_datetime(df["Date"])

start_hhmm = df["Heures"].str.split(" - ").str[0]

df["dt_local_naive"] = pd.to_datetime(
    df["Date"].dt.strftime("%Y-%m-%d") + " " + start_hhmm,
    format="%Y-%m-%d %H:%M",
    errors="raise",
)

df["dt_utc"] = (
    df["dt_local_naive"]
      .dt.tz_localize("Etc/GMT-1")
      .dt.tz_convert("UTC")
)

df = df.drop(columns=["dt_local_naive"])

In [38]:
df_reserves = df.copy()
display(df_reserves)

,Date,Heures,Type de réserve,Type de produit,Sens de la réserve,Quantité contractualisée (en MW),Prix de la réserve (en euros/MW/15min),Temporalité,dt_utc
0,2025-01-01,00:00 - 00:15,Réserve primaire,STD,A la hausse et à la baisse,670,0.69,Journalier,2024-12-31 23:00:00+00:00
1,2025-01-01,00:00 - 00:15,Réserve secondaire,STD,A la hausse,713,5.39,Journalier,2024-12-31 23:00:00+00:00
2,2025-01-01,00:00 - 00:15,Réserve secondaire,STD,A la baisse,755,5.39,Journalier,2024-12-31 23:00:00+00:00
3,2025-01-01,00:00 - 00:15,Réserve rapide,SPE0,A la hausse,405,0.03,Journalier,2024-12-31 23:00:00+00:00
4,2025-01-01,00:00 - 00:15,Réserve rapide,SPE0,A la hausse,651,0.31,Annuel,2024-12-31 23:00:00+00:00
...,...,...,...,...,...,...,...,...,...
199081,2025-12-14,00:00 - 00:15,Réserve secondaire,STD,A la hausse,701,4.92,Journalier,2025-12-13 23:00:00+00:00
199082,2025-12-14,00:00 - 00:15,Réserve secondaire,STD,A la baisse,758,3.64,Journalier,2025-12-13 23:00:00+00:00
199083,2025-12-14,00:00 - 00:15,Réserve rapide,SPE0,A la hausse,393,0.21,Journalier,2025-12-13 23:00:00+00:00
199084,2025-12-14,00:00 - 00:15,Réserve rapide,SPE0,A la hausse,607,0.31,Périodique,2025-12-13 23:00:00+00:00


In [39]:
df_reserves.rename(columns={"Prix de la réserve (en euros/MW/15min)": "Price",
                          "Quantité contractualisée (en MW)": "Quantity",
                          "dt_utc": "Datetime"}, inplace=True)

In [40]:
df_reserves.drop(columns=["Date", "Heures"], inplace=True)

In [47]:
display(df_reserves)

,Type de réserve,Type de produit,Sens de la réserve,Quantity,Price,Temporalité,Datetime,Way,Type
0,Réserve primaire,STD,A la hausse et à la baisse,670,0.69,Journalier,2024-12-31 23:00:00+00:00,UP_DOWN,FCR
1,Réserve secondaire,STD,A la hausse,713,5.39,Journalier,2024-12-31 23:00:00+00:00,UP,aFRR
2,Réserve secondaire,STD,A la baisse,755,5.39,Journalier,2024-12-31 23:00:00+00:00,DOWN,aFRR
6,Réserve primaire,STD,A la hausse et à la baisse,670,0.69,Journalier,2024-12-31 23:15:00+00:00,UP_DOWN,FCR
7,Réserve secondaire,STD,A la hausse,713,5.39,Journalier,2024-12-31 23:15:00+00:00,UP,aFRR
...,...,...,...,...,...,...,...,...,...
199075,Réserve secondaire,STD,A la hausse,701,2.60,Journalier,2025-12-13 22:45:00+00:00,UP,aFRR
199076,Réserve secondaire,STD,A la baisse,799,2.79,Journalier,2025-12-13 22:45:00+00:00,DOWN,aFRR
199080,Réserve primaire,STD,A la hausse et à la baisse,641,1.33,Journalier,2025-12-13 23:00:00+00:00,UP_DOWN,FCR
199081,Réserve secondaire,STD,A la hausse,701,4.92,Journalier,2025-12-13 23:00:00+00:00,UP,aFRR


In [42]:
df_reserves["Way"] = df_reserves["Sens de la réserve"].map({
    "A la hausse": "UP",
    "A la baisse": "DOWN",
    "A la hausse et à la baisse": "UP_DOWN"
})

In [44]:
df_reserves["Type"] = df_reserves["Type de réserve"].map({
    "Réserve primaire": "FCR",
    "Réserve secondaire": "aFRR",
})
df_reserves = df_reserves[df_reserves["Type"].isin(["FCR", "aFRR"])]
df_reserves = df_reserves[df_reserves["Type de produit"] == "STD"]

In [46]:
df_reserves = df_reserves[df_reserves["Temporalité"] == "Journalier"]

In [50]:
display(df_reserves)

,Datetime,Type,Way,Price
0,2024-12-31 23:00:00+00:00,FCR,UP_DOWN,0.69
1,2024-12-31 23:00:00+00:00,aFRR,UP,5.39
2,2024-12-31 23:00:00+00:00,aFRR,DOWN,5.39
6,2024-12-31 23:15:00+00:00,FCR,UP_DOWN,0.69
7,2024-12-31 23:15:00+00:00,aFRR,UP,5.39
...,...,...,...,...
199075,2025-12-13 22:45:00+00:00,aFRR,UP,2.60
199076,2025-12-13 22:45:00+00:00,aFRR,DOWN,2.79
199080,2025-12-13 23:00:00+00:00,FCR,UP_DOWN,1.33
199081,2025-12-13 23:00:00+00:00,aFRR,UP,4.92


In [49]:
df_reserves = df_reserves[["Datetime", "Type", "Way", "Price"]]

In [51]:
df_reserves.index = pd.to_datetime(df_reserves["Datetime"], utc=True)
df_reserves.drop(columns=["Datetime"], inplace=True)
display(df_reserves)

C:\Users\thibc\AppData\Local\Temp\ipykernel_23992\1755330604.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reserves.drop(columns=["Datetime"], inplace=True)


,Type,Way,Price
Datetime,,,
2024-12-31 23:00:00+00:00,FCR,UP_DOWN,0.69
2024-12-31 23:00:00+00:00,aFRR,UP,5.39
2024-12-31 23:00:00+00:00,aFRR,DOWN,5.39
2024-12-31 23:15:00+00:00,FCR,UP_DOWN,0.69
2024-12-31 23:15:00+00:00,aFRR,UP,5.39
...,...,...,...
2025-12-13 22:45:00+00:00,aFRR,UP,2.60
2025-12-13 22:45:00+00:00,aFRR,DOWN,2.79
2025-12-13 23:00:00+00:00,FCR,UP_DOWN,1.33


In [52]:
df_reserves.to_parquet("data/reserves.parquet")

In [1]:
import pandas as pd
df = pd.read_parquet("../data/reserves.parquet")
display(df)

,Type,Way,Price
Datetime,,,
2024-12-31 23:00:00+00:00,FCR,UP_DOWN,0.69
2024-12-31 23:00:00+00:00,aFRR,UP,5.39
2024-12-31 23:00:00+00:00,aFRR,DOWN,5.39
2024-12-31 23:15:00+00:00,FCR,UP_DOWN,0.69
2024-12-31 23:15:00+00:00,aFRR,UP,5.39
...,...,...,...
2025-12-13 22:45:00+00:00,aFRR,UP,2.60
2025-12-13 22:45:00+00:00,aFRR,DOWN,2.79
2025-12-13 23:00:00+00:00,FCR,UP_DOWN,1.33


In [2]:
wide = (
    df.assign(col=df["Type"] + "_" + df["Way"])
      .pivot_table(index="Datetime", columns="col", values="Price", aggfunc="last")
      .sort_index()
)
wide = wide.rename(columns={
    "FCR_UP_DOWN": "FCR"
})

In [4]:
wide = wide[wide.index.year == 2025]

In [5]:
display(wide)

col,FCR,aFRR_DOWN,aFRR_UP
Datetime,,,
2025-01-01 00:00:00+00:00,0.69,5.00,5.00
2025-01-01 00:15:00+00:00,0.69,5.00,5.00
2025-01-01 00:30:00+00:00,0.69,5.00,5.00
2025-01-01 00:45:00+00:00,0.69,5.00,5.00
2025-01-01 01:00:00+00:00,0.69,3.37,2.70
...,...,...,...
2025-12-13 22:00:00+00:00,0.55,2.79,2.60
2025-12-13 22:15:00+00:00,0.55,2.79,2.60
2025-12-13 22:30:00+00:00,0.55,2.79,2.60


In [6]:
wide.to_parquet("data/reserves.parquet")

In [2]:
import pandas as pd
df = pd.read_parquet("../data/prices.parquet")
display(df)

,Electricity
Datetime,
2015-01-01 00:00:00+00:00,36.56
2015-01-01 00:15:00+00:00,36.56
2015-01-01 00:30:00+00:00,36.56
2015-01-01 00:45:00+00:00,36.56
2015-01-01 01:00:00+00:00,36.56
...,...
2025-12-21 05:00:00+00:00,48.96
2025-12-21 05:15:00+00:00,48.96
2025-12-21 05:30:00+00:00,48.96


In [9]:
df.rename(columns={"Price": "Electricity"}, inplace=True)
df.to_parquet("data/prices.parquet")